# Playground · Regresión Logística

**Tópicos de Inteligencia de Negocios** · Clasificación binaria

> Mueve los sliders y observa cómo cambian la **frontera de decisión**, las **probabilidades** y las **métricas de clasificación** en tiempo real.

¿Qué vas a poder hacer aquí?

1. Generar datasets de clasificación binaria con distintas formas (linealmente separables, lunas, círculos).
2. Subir tu propio CSV con dos features numéricas y una etiqueta binaria.
3. Variar la regularización (C, L1/L2) y el umbral de decisión.
4. Ver el efecto en la **frontera de decisión**, **mapa de probabilidades**, **matriz de confusión** y **curva ROC**.

> 🔍 **Tip:** junto a cada control verás un botón azul **`?`**. Haz clic para desplegar una explicación. Cuando termines de configurar, presiona **🚀 Entrenar modelo**.


## Marco Teórico

### El modelo
Aplica la **función sigmoide** $\sigma$ a una combinación lineal de las features:

$$\hat{p} = \sigma(z) = \frac{1}{1 + e^{-z}}, \quad z = \theta_0 + \theta_1 x_1 + \theta_2 x_2 + \cdots$$

$\hat{p}$ es la **probabilidad** de pertenecer a la clase positiva. La predicción de clase se obtiene comparando con un **umbral** (default 0.5):

$$\hat{y} = \begin{cases} 1 & \text{si } \hat{p} \geq \text{umbral} \\ 0 & \text{si } \hat{p} < \text{umbral} \end{cases}$$

### Función de costo (Log-Loss / Entropía Cruzada)
$$J(\theta) = -\frac{1}{m} \sum_{i=1}^{m} \left[ y^{(i)} \log(\hat{p}^{(i)}) + (1-y^{(i)}) \log(1-\hat{p}^{(i)}) \right]$$

### Regularización
Como en regresión lineal, añadimos una penalización al costo:

| Tipo | Penalización |
|------|--------------|
| **L2 (Ridge)** | $\frac{1}{2C} \sum \theta_j^2$ |
| **L1 (Lasso)** | $\frac{1}{C} \sum \mid\theta_j\mid$ |

> ⚠️ **Cuidado con `C`:** scikit-learn usa `C = 1/λ`. Es decir, **C grande → poca regularización**, **C pequeño → mucha regularización**. Es el inverso de lo que estás acostumbrado.

### Métricas de clasificación

| Métrica | Fórmula | Útil cuando... |
|---------|---------|----------------|
| **Accuracy** | $\frac{VP+VN}{Total}$ | Las clases están balanceadas |
| **Precisión** | $\frac{VP}{VP+FP}$ | Quieres minimizar falsos positivos |
| **Recall** | $\frac{VP}{VP+FN}$ | Quieres detectar todos los positivos |
| **F1-Score** | $2 \cdot \frac{P \cdot R}{P+R}$ | Balance entre precisión y recall |
| **AUC-ROC** | área bajo la curva | Mide capacidad de discriminación |


## 1. Configuración inicial

In [ ]:
# ===== Auto-instalar paquetes que no vienen precargados en Pyodide (JupyterLite) =====
# Pyodide ya trae numpy, pandas, matplotlib y scikit-learn,
# pero ipywidgets hay que instalarlo en caliente la primera vez (~10 segundos).
try:
    import ipywidgets  # noqa: F401
except ImportError:
    import micropip  # disponible solo en Pyodide
    print('⏳ Instalando ipywidgets en el navegador (1 vez)...')
    await micropip.install('ipywidgets')
    print('✅ ipywidgets instalado')

import io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_classification, make_moons, make_circles, make_blobs
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, roc_curve, confusion_matrix)

import ipywidgets as widgets
from IPython.display import display, clear_output

plt.rcParams['figure.dpi'] = 90
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

print('✅ Listo. Librerías cargadas.')


## 2. Funciones auxiliares

In [ ]:
# UI helper: agrega un botón ? plegable junto a cualquier widget
def con_ayuda(control, explicacion):
    btn = widgets.Button(description='?', button_style='info', tooltip=explicacion,
                         layout=widgets.Layout(width='30px', height='28px', margin='0 0 0 4px'))
    panel = widgets.HTML(
        value=(f'<div style="background:#dbeafe; padding:8px 10px; border-radius:4px; '
               f'border-left:3px solid #2563eb; margin:2px 0 8px 18px; font-size:12px; '
               f'color:#1e3a8a;">💡 {explicacion}</div>'),
        layout=widgets.Layout(display='none'),
    )
    btn.on_click(lambda _: setattr(panel.layout, 'display',
                                   'none' if panel.layout.display != 'none' else 'block'))
    return widgets.VBox([widgets.HBox([control, btn]), panel])


def generar_dataset_clasificacion(tipo='Lunas', n=200, ruido=0.2, seed=42):
    """Genera datasets 2D para clasificación binaria."""
    if tipo == 'Lunas':
        X, y = make_moons(n_samples=n, noise=ruido, random_state=seed)
    elif tipo == 'Círculos':
        X, y = make_circles(n_samples=n, noise=ruido, factor=0.4, random_state=seed)
    elif tipo == 'Blobs (linealmente separables)':
        X, y = make_blobs(n_samples=n, centers=2, cluster_std=ruido*4 + 0.5,
                          random_state=seed)
    elif tipo == 'Classification (con ruido)':
        X, y = make_classification(n_samples=n, n_features=2, n_redundant=0,
                                   n_informative=2, n_clusters_per_class=1,
                                   flip_y=ruido*0.3, class_sep=2.0 - ruido,
                                   random_state=seed)
    return X, y


def construir_logistica(C, penalty, solver, max_iter=1000):
    """Devuelve un Pipeline con StandardScaler + LogisticRegression."""
    kwargs = dict(C=max(C, 1e-6), max_iter=max_iter, random_state=42)
    if penalty == 'none':
        kwargs['penalty'] = None
        # solver para None debe ser uno que lo soporte
        kwargs['solver'] = 'lbfgs'
    else:
        kwargs['penalty'] = penalty
        kwargs['solver'] = solver
        if penalty == 'elasticnet':
            kwargs['l1_ratio'] = 0.5
    return Pipeline([
        ('scaler', StandardScaler()),
        ('modelo', LogisticRegression(**kwargs)),
    ])


def calcular_metricas_clasif(y_true, y_pred, y_proba):
    """DataFrame con métricas estándar de clasificación binaria."""
    return pd.DataFrame({
        'Métrica': ['Accuracy', 'Precisión', 'Recall', 'F1', 'AUC-ROC'],
        'Valor': [
            accuracy_score(y_true, y_pred),
            precision_score(y_true, y_pred, zero_division=0),
            recall_score(y_true, y_pred, zero_division=0),
            f1_score(y_true, y_pred, zero_division=0),
            roc_auc_score(y_true, y_proba) if len(np.unique(y_true)) > 1 else float('nan'),
        ],
    }).round(4)


def graficar_resultados_clasif(X, y, pipeline, X_train, X_test, y_train, y_test,
                                umbral, titulo=''):
    """4 paneles: frontera + matriz confusión + ROC + métricas."""
    fig = plt.figure(figsize=(14, 8))
    gs = fig.add_gridspec(2, 3, hspace=0.35, wspace=0.3)
    ax1 = fig.add_subplot(gs[0, :2])  # frontera (grande)
    ax2 = fig.add_subplot(gs[0, 2])    # confusión
    ax3 = fig.add_subplot(gs[1, :2])   # ROC
    ax4 = fig.add_subplot(gs[1, 2])    # métricas (texto)

    # ----- 1. Frontera de decisión + heatmap de probabilidad -----
    margen = 0.5
    xx, yy = np.meshgrid(
        np.linspace(X[:,0].min()-margen, X[:,0].max()+margen, 200),
        np.linspace(X[:,1].min()-margen, X[:,1].max()+margen, 200))
    grid = np.c_[xx.ravel(), yy.ravel()]
    Z_proba = pipeline.predict_proba(grid)[:,1].reshape(xx.shape)
    cs = ax1.contourf(xx, yy, Z_proba, levels=20, cmap='RdBu_r', alpha=0.6, vmin=0, vmax=1)
    ax1.contour(xx, yy, Z_proba, levels=[umbral], colors='black', linewidths=2.0)
    ax1.scatter(X_train[y_train==0, 0], X_train[y_train==0, 1], c='#1d4ed8',
                s=40, alpha=0.8, edgecolor='white', linewidth=0.5, label='Train · Clase 0')
    ax1.scatter(X_train[y_train==1, 0], X_train[y_train==1, 1], c='#dc2626',
                s=40, alpha=0.8, edgecolor='white', linewidth=0.5, label='Train · Clase 1')
    ax1.scatter(X_test[y_test==0, 0], X_test[y_test==0, 1], c='#1d4ed8',
                s=70, marker='s', alpha=0.9, edgecolor='black', linewidth=1.0, label='Test · Clase 0')
    ax1.scatter(X_test[y_test==1, 0], X_test[y_test==1, 1], c='#dc2626',
                s=70, marker='s', alpha=0.9, edgecolor='black', linewidth=1.0, label='Test · Clase 1')
    ax1.set_title(f'Frontera de decisión (umbral = {umbral:.2f}) {titulo}',
                  fontsize=12, fontweight='bold')
    ax1.set_xlabel('Feature 1'); ax1.set_ylabel('Feature 2')
    ax1.legend(loc='best', fontsize=9, framealpha=0.9)
    plt.colorbar(cs, ax=ax1, label='P(clase=1)', fraction=0.04, pad=0.02)

    # ----- 2. Matriz de confusión (test) -----
    y_proba_te = pipeline.predict_proba(X_test)[:,1]
    y_pred_te = (y_proba_te >= umbral).astype(int)
    cm = confusion_matrix(y_test, y_pred_te, labels=[0, 1])
    ax2.imshow(cm, cmap='Blues', aspect='auto')
    for i in range(2):
        for j in range(2):
            ax2.text(j, i, cm[i,j], ha='center', va='center',
                     color='white' if cm[i,j] > cm.max()/2 else 'black',
                     fontsize=14, fontweight='bold')
    ax2.set_xticks([0,1]); ax2.set_yticks([0,1])
    ax2.set_xticklabels(['Pred 0','Pred 1'])
    ax2.set_yticklabels(['Real 0','Real 1'])
    ax2.set_title('Matriz de Confusión (Test)', fontsize=11, fontweight='bold')
    ax2.grid(False)

    # ----- 3. Curva ROC -----
    if len(np.unique(y_test)) > 1:
        fpr, tpr, _ = roc_curve(y_test, y_proba_te)
        auc = roc_auc_score(y_test, y_proba_te)
        ax3.plot(fpr, tpr, color='#2563eb', linewidth=2.5, label=f'AUC = {auc:.3f}')
        ax3.plot([0,1], [0,1], 'k--', alpha=0.5, label='Aleatorio')
        ax3.fill_between(fpr, tpr, alpha=0.15, color='#2563eb')
        ax3.set_xlabel('Tasa de Falsos Positivos (FPR)')
        ax3.set_ylabel('Tasa de Verdaderos Positivos (TPR)')
        ax3.set_title('Curva ROC', fontsize=11, fontweight='bold')
        ax3.legend(loc='lower right')
    else:
        ax3.text(0.5, 0.5, 'Solo una clase en test',
                 ha='center', va='center', transform=ax3.transAxes)
        ax3.set_title('Curva ROC')

    # ----- 4. Métricas como texto -----
    df_m = calcular_metricas_clasif(y_test, y_pred_te, y_proba_te)
    ax4.axis('off')
    txt = '\n'.join([f'{r["Métrica"]:<10s}: {r["Valor"]:.4f}'
                       for _, r in df_m.iterrows()])
    ax4.text(0.05, 0.95, '📊 Métricas (Test)', transform=ax4.transAxes,
             fontsize=11, fontweight='bold', va='top')
    ax4.text(0.05, 0.80, txt, transform=ax4.transAxes,
             fontsize=11, va='top', family='monospace')

    plt.tight_layout()
    plt.show()


## 3. Playground · Datos sintéticos 🧪

Genera datasets de clasificación binaria 2D y observa cómo la regresión logística traza una **frontera lineal** entre las clases. Podrás ver claramente cuándo el modelo es suficiente (datos linealmente separables) y cuándo no (lunas, círculos).

> 🔍 **Tip:** los botones **`?`** te explican cada parámetro. Cuando configures, presiona **🚀 Entrenar modelo**.


In [ ]:
# ----- Widgets DATOS -----
w_tipo = widgets.Dropdown(
    options=['Blobs (linealmente separables)', 'Lunas', 'Círculos', 'Classification (con ruido)'],
    value='Lunas', description='Dataset:', style={'description_width': '110px'})
w_n = widgets.IntSlider(value=200, min=50, max=600, step=10,
                        description='n puntos:', style={'description_width': '110px'})
w_ruido = widgets.FloatSlider(value=0.2, min=0.0, max=0.6, step=0.05,
                              description='Ruido:', style={'description_width': '110px'})
w_seed = widgets.IntSlider(value=42, min=0, max=100,
                           description='Semilla:', style={'description_width': '110px'})

# ----- Widgets MODELO -----
w_C = widgets.FloatLogSlider(value=1.0, base=10, min=-3, max=3,
                             description='C:', style={'description_width': '110px'})
w_pen = widgets.Dropdown(options=['l2', 'l1', 'elasticnet', 'none'],
                         value='l2', description='Penalty:',
                         style={'description_width': '110px'})
w_solver = widgets.Dropdown(options=['lbfgs', 'liblinear', 'saga'],
                            value='lbfgs', description='Solver:',
                            style={'description_width': '110px'})
w_thresh = widgets.FloatSlider(value=0.5, min=0.05, max=0.95, step=0.05,
                               description='Umbral:', style={'description_width': '110px'})
w_split = widgets.FloatSlider(value=0.25, min=0.1, max=0.5, step=0.05,
                              description='Test size:', style={'description_width': '110px'})

# ----- Explicaciones -----
ay_tipo = ('Forma del dataset. Blobs = separables con una recta. Lunas/Círculos = '
           'NO son separables linealmente, así que la regresión logística sufrirá. '
           '¡Pruébalos para ver las limitaciones del modelo!')
ay_n = 'Cantidad de puntos a generar. Más puntos = estimaciones más estables.'
ay_ruido = ('Cuánto se solapan las clases. Ruido bajo = clases bien separadas. '
            'Ruido alto = mucho overlap, difícil de clasificar.')
ay_seed = 'Misma semilla = mismos datos. Cámbiala para probar otro dataset igual de difícil.'
ay_C = ('Inverso de la fuerza de regularización. C alto = poca regularización (puede sobreajustar). '
        'C bajo = mucha regularización (puede subajustar). Es la C de 1/λ.')
ay_pen = ('Tipo de regularización. l2 (default) encoge coeficientes; l1 puede anularlos; '
          'elasticnet combina ambas; none no regulariza.')
ay_solver = ('Algoritmo de optimización. lbfgs es el default. liblinear soporta l1 y l2. '
             'saga soporta todas las penalizaciones (incluyendo elasticnet).')
ay_thresh = ('Umbral de decisión. P(clase=1) >= umbral → predice 1. Default 0.5. '
             'Súbelo si quieres ser más conservador con la clase 1 (más precisión, menos recall).')
ay_split = 'Proporción para test. 0.25 = 75% train / 25% test.'

panel_datos = widgets.VBox([
    widgets.HTML('<b>📊 Datos</b>  <span style="color:#64748b;font-size:11px;">'
                 '(clic en <b>?</b> para ayuda)</span>'),
    con_ayuda(w_tipo, ay_tipo), con_ayuda(w_n, ay_n),
    con_ayuda(w_ruido, ay_ruido), con_ayuda(w_seed, ay_seed),
])
panel_modelo = widgets.VBox([
    widgets.HTML('<b>🧮 Modelo</b>  <span style="color:#64748b;font-size:11px;">'
                 '(clic en <b>?</b> para ayuda)</span>'),
    con_ayuda(w_C, ay_C), con_ayuda(w_pen, ay_pen),
    con_ayuda(w_solver, ay_solver), con_ayuda(w_thresh, ay_thresh),
    con_ayuda(w_split, ay_split),
])
controles = widgets.HBox([panel_datos, panel_modelo])

salida = widgets.Output()

def actualizar():
    with salida:
        clear_output(wait=True)
        X, y = generar_dataset_clasificacion(w_tipo.value, w_n.value, w_ruido.value, w_seed.value)
        X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=w_split.value,
                                                   random_state=42, stratify=y)
        try:
            pipe = construir_logistica(w_C.value, w_pen.value, w_solver.value)
            pipe.fit(X_tr, y_tr)
        except Exception as e:
            print(f'⚠️ Error: {e}')
            print('Tip: algunas combinaciones penalty/solver no son compatibles. '
                  'Prueba lbfgs+l2, liblinear+l1, saga+elasticnet.')
            return
        graficar_resultados_clasif(X, y, pipe, X_tr, X_te, y_tr, y_te,
                                    w_thresh.value,
                                    titulo=f'· {w_tipo.value} · C={w_C.value:.2f} · {w_pen.value}')

# ----- Botón Entrenar e indicador de estado -----
btn_entrenar = widgets.Button(description='🚀 Entrenar modelo', button_style='primary',
                              layout=widgets.Layout(width='220px', height='40px',
                                                    margin='10px 0 6px 0'))
estado = widgets.HTML(value='<span style="color:#64748b;font-style:italic;">'
                            'Configura los parámetros y haz clic en <b>Entrenar modelo</b>.</span>')

def _stale(*_):
    estado.value = ('<span style="color:#ea580c;">🔄 <b>Cambios sin aplicar.</b> '
                    'Haz clic en <b>Entrenar modelo</b> para verlos reflejados.</span>')

def _go(_):
    btn_entrenar.disabled = True
    btn_entrenar.description = '⏳ Entrenando...'
    estado.value = '<span style="color:#2563eb;">⏳ Entrenando...</span>'
    try:
        actualizar()
        estado.value = ('<span style="color:#16a34a;">✅ <b>Modelo entrenado.</b> '
                        'Cambia parámetros y vuelve a entrenar.</span>')
    except Exception as e:
        estado.value = f'<span style="color:#dc2626;">❌ {e}</span>'
    finally:
        btn_entrenar.disabled = False
        btn_entrenar.description = '🚀 Entrenar modelo'

btn_entrenar.on_click(_go)
for w in [w_tipo, w_n, w_ruido, w_seed, w_C, w_pen, w_solver, w_thresh, w_split]:
    w.observe(_stale, names='value')

display(controles, btn_entrenar, estado, salida)
actualizar()
estado.value = ('<span style="color:#16a34a;">✅ <b>Modelo entrenado con la configuración inicial.</b> '
                'Modifica los sliders y dale a <b>Entrenar modelo</b>.</span>')


## 4. Playground · Sube tu CSV 📂

Sube un CSV con dos columnas numéricas (features) y una columna binaria (etiqueta 0/1).


In [ ]:
estado_csv = {'df': None}

w_upload = widgets.FileUpload(accept='.csv', multiple=False, description='📁 Subir CSV')
w_x1 = widgets.Dropdown(options=[], description='Feature 1:', style={'description_width': '110px'})
w_x2 = widgets.Dropdown(options=[], description='Feature 2:', style={'description_width': '110px'})
w_yc = widgets.Dropdown(options=[], description='Etiqueta:', style={'description_width': '110px'})

w_C2 = widgets.FloatLogSlider(value=1.0, base=10, min=-3, max=3,
                              description='C:', style={'description_width': '110px'})
w_pen2 = widgets.Dropdown(options=['l2', 'l1', 'elasticnet', 'none'], value='l2',
                          description='Penalty:', style={'description_width': '110px'})
w_solver2 = widgets.Dropdown(options=['lbfgs', 'liblinear', 'saga'], value='lbfgs',
                             description='Solver:', style={'description_width': '110px'})
w_thresh2 = widgets.FloatSlider(value=0.5, min=0.05, max=0.95, step=0.05,
                                description='Umbral:', style={'description_width': '110px'})
w_split2 = widgets.FloatSlider(value=0.25, min=0.1, max=0.5, step=0.05,
                               description='Test size:', style={'description_width': '110px'})

salida_csv = widgets.Output()
salida_info = widgets.Output()

def on_upload(change):
    with salida_info:
        clear_output(wait=True)
        if not w_upload.value:
            return
        try:
            archivo = w_upload.value[0] if isinstance(w_upload.value, tuple) else next(iter(w_upload.value.values()))
            contenido = archivo['content']
            nombre = archivo.get('name', 'archivo.csv')
        except Exception:
            archivo = list(w_upload.value.values())[0]
            contenido = archivo['content']
            nombre = archivo.get('metadata', {}).get('name', 'archivo.csv')
        try:
            df = pd.read_csv(io.BytesIO(bytes(contenido)))
        except Exception as e:
            print(f'⚠️ {e}'); return
        estado_csv['df'] = df
        cols_num = df.select_dtypes(include=[np.number]).columns.tolist()
        if len(cols_num) < 3:
            print(f'⚠️ Necesitas al menos 3 columnas numéricas (2 features + 1 etiqueta). '
                  f'Encontré: {cols_num}')
            return
        w_x1.options = cols_num; w_x2.options = cols_num; w_yc.options = cols_num
        w_x1.value = cols_num[0]; w_x2.value = cols_num[1]; w_yc.value = cols_num[-1]
        print(f'✅ Cargado: {nombre} — {df.shape[0]} filas × {df.shape[1]} columnas')
        print('\n📋 Vista previa:')
        display(df.head())

w_upload.observe(on_upload, names='value')

def actualizar2():
    with salida_csv:
        clear_output(wait=True)
        df = estado_csv['df']
        if df is None:
            print('⬆️ Sube un CSV primero.'); return
        try:
            sub = df[[w_x1.value, w_x2.value, w_yc.value]].dropna()
            X = sub[[w_x1.value, w_x2.value]].values
            y = sub[w_yc.value].values
            # Coerce a binario 0/1
            unicos = np.unique(y)
            if len(unicos) != 2:
                print(f'⚠️ La etiqueta debe ser binaria (2 clases). Encontré: {unicos}')
                return
            y = (y == unicos[1]).astype(int)
            X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=w_split2.value,
                                                       random_state=42, stratify=y)
            pipe = construir_logistica(w_C2.value, w_pen2.value, w_solver2.value)
            pipe.fit(X_tr, y_tr)
            graficar_resultados_clasif(X, y, pipe, X_tr, X_te, y_tr, y_te,
                                        w_thresh2.value,
                                        titulo=f'· {w_yc.value}')
        except Exception as e:
            print(f'⚠️ {e}')

btn_e2 = widgets.Button(description='🚀 Entrenar modelo', button_style='primary',
                        layout=widgets.Layout(width='220px', height='40px',
                                              margin='10px 0 6px 0'))
estado2 = widgets.HTML(value='<span style="color:#64748b;font-style:italic;">'
                             'Sube un CSV y haz clic en <b>Entrenar modelo</b>.</span>')

def _stale2(*_):
    if estado_csv['df'] is None: return
    estado2.value = ('<span style="color:#ea580c;">🔄 <b>Cambios sin aplicar.</b> '
                     'Haz clic en <b>Entrenar modelo</b>.</span>')

def _go2(_):
    if estado_csv['df'] is None:
        estado2.value = '<span style="color:#dc2626;">⚠️ Primero sube un CSV.</span>'
        return
    btn_e2.disabled = True; btn_e2.description = '⏳ Entrenando...'
    estado2.value = '<span style="color:#2563eb;">⏳ Entrenando...</span>'
    try:
        actualizar2()
        estado2.value = ('<span style="color:#16a34a;">✅ <b>Modelo entrenado.</b></span>')
    except Exception as e:
        estado2.value = f'<span style="color:#dc2626;">❌ {e}</span>'
    finally:
        btn_e2.disabled = False; btn_e2.description = '🚀 Entrenar modelo'

btn_e2.on_click(_go2)
for w in [w_x1, w_x2, w_yc, w_C2, w_pen2, w_solver2, w_thresh2, w_split2]:
    w.observe(_stale2, names='value')

panel_carga = widgets.VBox([
    widgets.HTML('<b>📁 Datos (CSV)</b>'),
    con_ayuda(w_upload, 'Sube tu .csv. Necesitas 2 columnas numéricas para features y 1 para la etiqueta binaria.'),
    con_ayuda(w_x1, 'Primera variable predictora.'),
    con_ayuda(w_x2, 'Segunda variable predictora.'),
    con_ayuda(w_yc, 'Variable a predecir (debe tener 2 valores únicos).'),
])
panel_modelo2 = widgets.VBox([
    widgets.HTML('<b>🧮 Modelo</b>'),
    con_ayuda(w_C2, ay_C), con_ayuda(w_pen2, ay_pen),
    con_ayuda(w_solver2, ay_solver), con_ayuda(w_thresh2, ay_thresh),
    con_ayuda(w_split2, ay_split),
])
display(widgets.HBox([panel_carga, panel_modelo2]), btn_e2, estado2, salida_info, salida_csv)


## 5. Ejercicios guiados 📝

### Ejercicio 1 — Lo que SÍ y lo que NO puede la regresión logística
1. Dataset = **Blobs** (linealmente separables), n = 200, ruido = 0.2.
2. Entrena con C = 1, penalty = l2. Observa la frontera y el AUC.
3. Cambia a Dataset = **Lunas**. Vuelve a entrenar.
4. **Pregunta:** ¿Por qué el AUC bajó tanto? ¿Qué tipo de frontera necesitarías?

### Ejercicio 2 — El umbral importa
1. Con Lunas y C=1, fija el umbral en 0.3, luego 0.5, luego 0.7.
2. **Pregunta:** ¿Cómo cambian Precisión vs Recall conforme subes el umbral? ¿En qué tipo de problema usarías umbral bajo (≤0.3) y en cuál uno alto (≥0.7)?

### Ejercicio 3 — Efecto de C (regularización)
1. Dataset = **Classification (con ruido)**, ruido = 0.4.
2. Prueba C = 0.001, luego 1, luego 1000.
3. **Pregunta:** ¿Cómo cambia la frontera con C muy chico vs muy grande?

### Ejercicio 4 — L1 vs L2
1. Mismo dataset del ejercicio 3.
2. Cambia entre penalty = l2 y penalty = l1 (con solver = liblinear o saga).
3. **Pregunta:** ¿Cuál tiende a producir coeficientes en cero? Justifica con la teoría que vieron en clase.

### Ejercicio 5 — Tu propio CSV
Sube un dataset binario (puede ser uno público como Titanic, breast cancer, etc.). Encuentra una pareja de features que dé un AUC > 0.7.


## 6. Resumen

- La regresión logística traza **fronteras lineales** entre clases — falla con datos no linealmente separables.
- El parámetro **C es el inverso** de la regularización: C alto = poca regularización.
- El **umbral** es un hiperparámetro de negocio: ajústalo según si te duele más un FP o un FN.
- Para clases desbalanceadas, **AUC** es más informativo que accuracy.
- Si tus datos no son separables linealmente, considera: features polinomiales, kernels, árboles, o redes.

---

> *Tópicos de Inteligencia de Negocios · Playground de Machine Learning*
